In [ ]:
import pandas as pd
import numpy as np

# preprocessing
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from imblearn.over_sampling import RandomOverSampler

# model machine learning
from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,log_loss
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
df = pd.read_csv('../0.dataset/dataset_Heart_attack_clean.csv',nrows=3000)
df_x = df.drop(columns='heart_attack')
df_y = df['heart_attack']

# 1. Binary Classification Random Forest

In [ ]:
feature_numerik = ['age','body_mass_index','systolic_blood_pressure','forced_expiratory_volume_1','time_to_event_or_censoring']
feature_categori = ['gender']

preprocessor = ColumnTransformer(
    transformers=[
        ('numerik_scaler',StandardScaler(),feature_numerik),
        ('categori_encoding',OneHotEncoder(),feature_categori)
        ],
   remainder='passthrough' # Kolom kategori yang sudah ter-encode dari sananya akan aman lewat lewat sini
)

((699,), (699,))

In [ ]:
selector_model = RandomForestClassifier(random_state=42)
model_RandomForestClassifier = Pipeline([
    ('preprocessing',preprocessor),
    ('oversampling',RandomOverSampler(random_state=42)),
    ('feature_selection',SequentialFeatureSelector(estimator=selector_model,direction='forward')),
    ('model_classifier',RandomForestClassifier(random_state=42))
])

In [ ]:
params = {
    'feature_selection__n_features_to_select': [5],
    'model_classifier__n_estimators': [int(x) for x in np.linspace(start=20, stop=40, num=5)],
    'model_classifier__criterion': ['gini', 'entropy'],
    'model_classifier__max_depth': [7, 10, 12, 15],
    'model_classifier__min_samples_split': [24, 25, 30],
    'model_classifier__min_samples_leaf': [20, 25, 30],
    'model_classifier__max_features': ['sqrt', 'log2'],
    'model_classifier__bootstrap': [True],
    'model_classifier__class_weight': ['balanced', None]
}
X_train,X_test,y_train,y_test = train_test_split(df_x,df_y,test_size=0.2,random_state=42)

random_search = RandomizedSearchCV(estimator=model_RandomForestClassifier,param_distributions=params,n_iter=5,cv=5,scoring='accuracy',random_state=42,n_jobs=-1,verbose=3)
random_search.fit(X_train,y_train)


Hyperparameter Terbaik: {'n_estimators': 20, 'min_samples_split': 25, 'min_samples_leaf': 25, 'max_features': 'log2', 'max_depth': 15, 'criterion': 'entropy', 'class_weight': 'balanced', 'bootstrap': True}

Akurasi Validasi Terbaik: 81.12

Fitur Terbaik Yang Terpilih:
['Usia', 'Tipe_Nyeri_Dada', 'Detak_Jantung_Max', 'Angina_Olahraga', 'Oldpeak_ST']


In [ ]:
best_model_pipeline = random_search.best_estimator_
preprocessor_step = best_model_pipeline.named_steps['preprocessing']
sfs_step = best_model_pipeline.named_steps['feature_selection']

kolom_setelah_transformasi = preprocessor_step.get_feature_names_out()
fitur_terpilih = kolom_setelah_transformasi[sfs_step.get_support()]

print(f'\nHyperparameter Terbaik:\n{random_search.best_params_}')
print(f'\nFitur Terbaik Yang Terpilih:\n{list(fitur_terpilih)}')

#cek score
tes_accuracy = best_model_pipeline.score(X_test, y_test)

y_pred_test = best_model_pipeline.predict(X_test)
y_prob_test = best_model_pipeline.predict_proba(X_test)

y_pred_train = best_model_pipeline.predict(X_train)
y_prob_train = best_model_pipeline.predict_proba(X_train)

print(f'\nAkurasi Validasi Terbaik: {random_search.best_score_*100:.2f}%')
print(f'\nAkurasi Data Test: {tes_accuracy*100:.2f}%')


Akurasi Pada Data Test: 74.00

Akurasi Pada Data Train: 78.47


In [46]:
def evaluate_model(pred,test,prob,evaluate,model_name='Decision Tree'):
    accuracy = accuracy_score(test,pred)
    precision = precision_score(test,pred)
    recall = recall_score(test,pred)
    f1 = f1_score(test,pred)
    roc_auc = roc_auc_score(test,prob[:,1])
    logloss = log_loss(test,prob)

    data = {
        'Model': [model_name],
        'Evaluated': [evaluate],
        'Accuracy': [accuracy],
        'Precision': [precision],
        'Recall': [recall],
        'F1-Score': [f1],
        'ROC-AUC': [roc_auc],
        'Log Loss': [logloss]
    }

    df_result = pd.DataFrame(data)
    return df_result

In [47]:
def check_model_fit(df_eval_train,df_eval_test):
    df_combined = pd.concat([df_eval_train,df_eval_test],ignore_index=True)
    accuracy_train = df_eval_train['Accuracy'].values[0]
    accuracy_test = df_eval_test['Accuracy'].values[0]
    gap = accuracy_train - accuracy_test

    if accuracy_train < 0.60 and accuracy_test <0.60:
        status = 'Undeerfitting (Akurasi Rendah)'
    elif gap > 0.05:
        status = f'Overfitting (gap:{gap*100:.2f})'
    elif gap < -0.05:
        status = 'Overfitting / Data Leakage (Test > Train)'
    else:
        status = 'Good Fit'

    df_combined['Status'] = status
    return df_combined

In [48]:
df_eval_train = evaluate_model(y_pred_train,y_train,y_prob_train,evaluate='Training')
df_eval_test = evaluate_model(y_pred_test,y_test,y_prob_test,evaluate='Test')
df_eval = check_model_fit(df_eval_train,df_eval_test)

print('================================= PREDIKSI DENGAN SAMPLE DATASET ===================================')
print(df_eval.to_string())
print("\n" + "="*100 + "\n")

================================= PREDIKSI DENGAN SAMPLE DATASET ===================================
           Model Evaluated  Accuracy  Precision    Recall  F1-Score   ROC-AUC  Log Loss    Status
0  Decision Tree  Training  0.784692   0.771858  0.808298  0.789658  0.884318  0.446501  Good Fit
1  Decision Tree      Test  0.740000   0.796407  0.751412  0.773256  0.833035  0.511175  Good Fit




In [ ]:
data_4_pasien = {
'gender': ['F', 'F', 'F', 'F'],
    'age': [31, 72, 28, 63],
    'body_mass_index': [25.9, 28.6, 27.0, 27.0],
    'smoker': [0, 0, 0, 0],  # Diasumsikan dari data numerik biner Anda
    'systolic_blood_pressure': [112.0, 125.0, 102.0, 130.0],
    'hypertension_treated': [1, 1, 0, 0],
    'family_history_of_cardiovascular_disease': [0, 0, 0, 0],
    'atrial_fibrillation': [0, 0, 0, 0],
    'chronic_kidney_disease': [0, 1, 0, 0],
    'rheumatoid_arthritis': [0, 0, 0, 0],
    'diabetes': [0, 0, 0, 0],
    'chronic_obstructive_pulmonary_disorder': [0, 0, 0, 0],
    'forced_expiratory_volume_1': [90.96, 88.00, 98.37, 100.00], # Mengisi nilai representatif berdasarkan teks data Anda
    'time_to_event_or_censoring': [9833, 100, 6954, 100],        # Mengisi nilai representatif berdasarkan teks data Anda
    'heart_attack': [1, 0, 1, 0]
}
data_pasien_baru_x = pd.DataFrame(data_4_pasien)
data_pasien_baru_y = data_pasien_baru_x['heart_attack']

,Usia,Tipe_Nyeri_Dada,Detak_Jantung_Max,Angina_Olahraga,Oldpeak_ST
0,1.458427,-0.30,-0.535966,44,-1.173811
1,-0.625040,1.20,1.495518,60,0.391270
2,0.648190,0.08,-0.611606,56,0.939049
3,-1.435277,-1.05,0.836366,75,1.056430
4,-0.046299,-0.48,-1.184312,14,-1.212938


In [ ]:
print("=== Melakukan Prediksi Data Pasien Baru ===")
prediksi_pasien = best_model_pipeline.predict(data_pasien_baru_x)
probabilitas_pasien = best_model_pipeline.predict_proba(data_pasien_baru_x)

hasil_df = data_pasien_baru_x.copy()
hasil_df['Prediksi Serangan Jantung'] = prediksi_pasien
hasil_df['Peluang Aman(%)'] = probabilitas_pasien[:,0] * 100
hasil_df['Peluang Terkena (%)'] = probabilitas_pasien[:,1] * 100

akurasi_bawaan = best_model_pipeline.score(data_pasien_baru_x, data_pasien_baru_y)
df_eval_data_baru = evaluate_model(
    pred=prediksi_pasien, 
    test=data_pasien_baru_y, 
    prob=probabilitas_pasien, 
    evaluate='Data_Baru'
)

print(f"Akurasi Model: {akurasi_bawaan * 100:.2f}%")
print("\nTabel Skor Evaluasi Lengkap Data Baru:")
print(df_eval_data_baru.to_string(index=False))
hasil_df[['Prediksi Serangan Jantung', 'Peluang Aman(%)', 'Peluang Terkena (%)']]

=== Melakukan Prediksi Data Pasien Baru ===
Akurasi Model: 80.00%

Tabel Skor Evaluasi Lengkap Data Baru:
        Model Evaluated  Accuracy  Precision  Recall  F1-Score  ROC-AUC  Log Loss
Decision Tree Data_Baru       0.8       0.75     1.0  0.857143      1.0   0.38456


,Peluang Aman(%),Peluang Terkena (%),Prediksi Serangan Jantung
0,17.495679,82.504321,1
1,71.963291,28.036709,0
2,2.712032,97.287968,1
3,31.225172,68.774828,1
4,18.943621,81.056379,1


# 2.Multiclass Classification Random Forest